In [ ]:
!pip install azure-storage-blob tqdm

import os
from google.colab import drive, userdata
from azure.storage.blob import BlobServiceClient
from tqdm.auto import tqdm

print("Đang kết nối với Google Drive...")
drive.mount('/content/drive')


Đang kết nối với Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

try:
    AZURE_CONNECTION_STRING = userdata.get('AZURE_CONNECTION_STRING')
except userdata.SecretNotFoundError:
    print("LỖI: Chưa tìm thấy secret 'AZURE_CONNECTION_STRING'")
    raise

CONTAINER_NAME = "data"
AZURE_FOLDER_PREFIX = "embedding_data/"
DRIVE_BASE_PATH = "/content/drive/MyDrive/Reddit_Project"

print("\nĐang kết nối với hệ thống máy chủ Azure")
blob_service_client = BlobServiceClient.from_connection_string(AZURE_CONNECTION_STRING)
container_client = blob_service_client.get_container_client(CONTAINER_NAME)

print("Đang quét danh sách file trên Azure")
blobs = list(container_client.list_blobs(name_starts_with=AZURE_FOLDER_PREFIX))
total_files = len(blobs)
print(f"Tìm thấy tổng cộng {total_files} file cần chuyển.")

print("\nBắt đầu chuyển dữ liệu từ Azure sang Google Drive:\n")

for blob in tqdm(blobs, desc="Tiến độ Tải", unit="file"):
    if not blob.name.endswith(".parquet"):
        continue

    local_file_path = os.path.join(DRIVE_BASE_PATH, blob.name)

    os.makedirs(os.path.dirname(local_file_path), exist_ok=True)

    if os.path.exists(local_file_path) and os.path.getsize(local_file_path) == blob.size:
        continue

    with open(local_file_path, "wb") as f:
        download_stream = container_client.download_blob(blob)
        f.write(download_stream.readall())

print("\n Đã chuyển dữ liệu xong")
print(f"Dữ liệu được lưu tại: {os.path.join(DRIVE_BASE_PATH, AZURE_FOLDER_PREFIX)}")


Đang kết nối với hệ thống máy chủ Azure
Đang quét danh sách file trên Azure
Tìm thấy tổng cộng 18051 file cần chuyển.

Bắt đầu chuyển dữ liệu từ Azure sang Google Drive:



Tiến độ Tải:   0%|          | 0/18051 [00:00<?, ?file/s]


 Đã chuyển dữ liệu xong
Dữ liệu được lưu tại: /content/drive/MyDrive/Reddit_Project/embedding_data/
